# UNet denoiser HPO: economical search and seed confirmation

The same resumable workflow as `DiT_Generation_HPO_Runner.ipynb`, using the
repository's conditional convolutional **UNet** and shared `common.hpo` API.
Optimize **validation EMA noise MSE**, with TensorBoard, persistent Optuna storage,
performance/OOM pruning, frozen finalists and paired fresh-seed confirmations.

**Default budget: one A100 80 GB, at most 24 hours of experiment time.** Search
gets up to 18 hours and confirmation retains 6 hours. The targets are 12 successful
trials for review and 60 for the main search, with an optional extension to 100.
They are ceilings, not a promise of completion. Every trial retains at most
50 epochs and early-stopping patience 5. Two finalists receive three fresh seeds
each. No exhaustive search or globally best model is claimed.

**Data:** CIFAR-10, with a seeded 20% holdout from the official training split
(approximately 40,000 training / 10,000 validation images). The official test
set is excluded from selection. This intentionally improves the validation
protocol over the reference notebook's test-guided search. To reproduce that
protocol, set `VALIDATION_SOURCE="test"`, `VALIDATION_RATIO=0.0` before starting
a fresh study; its test scores then become tuning scores, not held-out estimates.

The notebook is delivered unexecuted. Run it only in an online GPU container.
Run cells individually if you want to inspect the pilot before the main search.
Run All starts the full bounded search and confirmations; the optional extension
is disabled unless explicitly enabled below.

## Runtime and cost controls

Use the complete matching checkout with **TensorFlow 2.20.0 / Keras 3.11.2**.
Select Python 3 (ipykernel), or a kernel containing those versions, and restart
before setup. The coordinator hides GPUs; isolated admitted children train.
Container setup verifies packages without replacing them. The inherited hosted
bootstrap can install the pinned environment on Colab/Kaggle, but this notebook
and its matching helpers must be present in the checkout before use.

Start with `GPU_IDS=[0]`, `CONCURRENT_TRIALS=1`, and a **24 GiB** TensorFlow cap.
This is a conservative requested reservation, **not a measured UNet capacity
certificate**. Admission adds per-worker overhead and device headroom, checks
process ownership and preserves existing jobs. Unknown owners block execution.
Recognized search OOMs are recorded and pruned, never silently downsized.

After representative large models fit, two workers on one 80 GB GPU may improve
throughput, but benchmark before increasing concurrency. The DiT notebook's
17-workers-per-H100 result does not apply to UNet. More GPUs reduce waiting time
only when their observed throughput justifies the extra rental cost. Confirmations
use one worker per selected GPU. VRAM is never pooled across devices.

The persistent clock starts at first search, includes pauses/restarts, and cannot
be extended by rerunning a cell. Setup precedes that clock. Save the entire
RESULTS_PATH, including SQLite, sampler state, configs and notebook_runner.
Use a new directory after scientific settings, source or package versions change.

In [ ]:
import os
import platform
import sys

from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Vision-Transformers"
REPOSITORY_NAME = "continual_learning_with_diffusion_vision_transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{REPOSITORY_NAME}.git"
REVISION = "main"

# Reject laptop kernels before cloning a checkout or installing any packages.
if platform.system() != "Linux":
    raise RuntimeError("Use an online GPU runtime; laptop execution is prohibited.")
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    RUNTIME = "kaggle"
elif os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
    RUNTIME = "colab"
elif Path("/.dockerenv").is_file() or Path("/run/.containerenv").is_file():
    RUNTIME = "container"
else:
    raise RuntimeError("Use Colab, Kaggle, or a remote Linux GPU container.")

# GPU work happens only in admitted children, never in this coordinator kernel.
if "tensorflow" in sys.modules or "keras" in sys.modules:
    raise RuntimeError("Restart the kernel and run setup before importing TensorFlow or Keras.")
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

os.environ["CONTINUAL_RUNTIME"] = RUNTIME
# Reuse the same standard-library initializer as the other online notebooks.
_locations = (
    Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME, 
    Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME
)
_initializer = next((
    path / "files" / "notebooks" / "init.py" for path in _locations
    if (path / "files" / "notebooks" / "init.py").is_file()
), None)
_url = (
    f"https://raw.githubusercontent.com/hosein-fanai/{REPOSITORY_NAME}/"
    f"{REVISION}/files/notebooks/init.py"
)
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
CHECKOUT_ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, 
    revision=REVISION, 
    runtime="local" if RUNTIME == "container" else RUNTIME, 
    cuda=False, 
    install=False if RUNTIME == "container" else None
)

os.chdir(CHECKOUT_ROOT)
if str(CHECKOUT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHECKOUT_ROOT))
os.environ.setdefault("KERAS_HOME", str(CHECKOUT_ROOT.parent / ".keras"))

from common.dit_hpo_remote import inspect_remote
from common.dit_hpo_runner import (
    make_plan, search_summary, run_search, freeze_finalists, 
    run_confirmations, confirmation_summary, budget_summary
)


runtime = inspect_remote(CHECKOUT_ROOT)
display({key: value for key, value in runtime.items() if key not in {"source_sha256", "worker_policy", "concurrent_trials"}})
print("Fingerprinted source files:", len(runtime["source_sha256"]))

In [ ]:
from copy import deepcopy

from common.unet_hpo import SEARCH_SPACE_OVERRIDES as UNET_SEARCH_SPACE


DATASET = "CIFAR10"
RESULTS_PATH = "files/results/unet_generation_hpo_v1"
VALIDATION_SOURCE = "split"
VALIDATION_RATIO = 0.2
GPU_IDS = [0]
CONCURRENT_TRIALS = 1
WORKER_GPU_MEMORY_LIMIT_MB = 24576
EPOCHS = 50
N_STARTUP_TRIALS = 12
SEARCH_SEED = 42
SEARCH_SPACE_OVERRIDES = deepcopy(UNET_SEARCH_SPACE)

PRUNING = {
    "type": "percentile", 
    "monitor": "val_noise_loss", 
    "percentile": 50.0, 
    "n_startup_trials": 12, 
    "n_warmup_steps": 9, 
    "interval_steps": 5, 
    "n_min_trials": 5
}

REVIEW_TARGET = 12
SEARCH_TARGET = 60
EXTENSION_TARGET = 100
ENABLE_EXTENSION = False
MAX_ATTEMPTS = 240
BATCH_TRIALS = max(12, 2 * CONCURRENT_TRIALS)
EXPERIMENT_HOURS = 24.0
CONFIRMATION_RESERVE_HOURS = 6.0
TOP_K = 2
CONFIRMATION_SEEDS = [101, 202, 303]

plan = make_plan(
    CHECKOUT_ROOT, RESULTS_PATH, 
    dataset_name=DATASET, epochs=EPOCHS, n_startup_trials=N_STARTUP_TRIALS, 
    concurrent_trials=CONCURRENT_TRIALS, gpu_ids=GPU_IDS, pruning=PRUNING, 
    validation_source=VALIDATION_SOURCE, validation_ratio=VALIDATION_RATIO, 
    experiment_hours=EXPERIMENT_HOURS, 
    confirmation_reserve_hours=CONFIRMATION_RESERVE_HOURS, 
    search_space_overrides=SEARCH_SPACE_OVERRIDES, 
    worker_gpu_memory_limit_mb=WORKER_GPU_MEMORY_LIMIT_MB, 
    model_name="unet", seed=SEARCH_SEED
)
display(plan["hpo"])
print("Persistent study:", plan["study_root"])
TENSORBOARD_DIRECTORY = Path(plan["study_root"])
display(plan["identity"]["gpus"])
display(budget_summary(plan))

## Thorough, conditional search space

All settings flow through the maintained API. The explicit `denoiser_v1` switch
leaves older UNet studies' distributions unchanged. This is a broad finite domain
sampled by TPE, not a Cartesian grid. Limited runs cannot cover every interaction.

| Group | Domain and condition |
| --- | --- |
| Encoder widths | 32-64; 32-64-96; 32-64-128; 48-96-192; 64-96-128; 64-128-256; 32-64-128-256 |
| Residual depth | 1 / 2 / 3 blocks at each encoder/decoder level |
| Bottleneck | 1 / 1.5 / 2 times widest encoder width; 1 / 2 / 3 blocks |
| Embedding channels | Total 64 / 96 / 128 / 192; balanced or time-rich allocation |
| Nonlinearity | swish / gelu / relu |
| Normalization | Batch normalization enabled / disabled; no unsupported GroupNorm knob |
| Spatial dropout | 0 / 0.05 / 0.1 / 0.2 |
| Downsampling | average pooling / max pooling / learned strided convolution |
| Upsampling | interpolation / convolution plus interpolation / transpose convolution |
| Interpolation | nearest / bilinear; sampled only for interpolation-based upsampling |
| Optimizer | Adam / AdamW |
| Learning rate | 1e-4 to 1e-3, log-uniform; cosine decay |
| Weight decay | 1e-6 to 1e-3, log-uniform; AdamW only |
| Batch size | 32 / 64 / 128 |
| Gradient clipping | Global norm None / 1 / 5; per-variable clipping disabled |
| EMA decay | 0.995 / 0.999 |
| CFG label dropout | 0.05 / 0.1 / 0.2; class conditioning and CFG remain enabled |

Fixed controls keep the objective comparable: ordinary epsilon prediction,
**1,000 timesteps, clipped-cosine noise schedule, MSE, linear output, encoder
skips enabled, zero auxiliary image loss**. Different noise schedules change
the prediction task, so their raw MSEs should not be treated as equivalent
architecture scores. Compare schedules later in separate controlled studies
using a common image-quality evaluation. Model constructors do not expose
attention or convolution-kernel choices here; those are not fictitious HPO axes.

The 2–4 downsamplings divide 32 evenly, reaching 8, 4 or 2 pixels. Width templates
avoid invalid or needlessly huge hierarchies. A balanced embedding allocates
roughly one third to image/time/label; time-rich allocates roughly one quarter,
one half, one quarter, preserving the selected total exactly.

Sampling steps, guidance scale and eta do not change this loss-based HPO objective,
so they are not searched. Existing reports use fixed sample settings. Tune those
only after selecting a denoiser, and assess image quality/diversity and downstream
replay usefulness separately. A better denoising score need not mean better images.

## Pruning, recovery and TensorBoard

TPE begins with 12 startup observations. The median/50th-percentile rule starts
after 12 completed reference trials, no earlier than epoch 10, then every five
epochs when at least five reference observations exist. This economical policy
can discard slow starters; finalists keep the same 50-epoch limit and early
stopping but disable performance pruning. Numerical/OOM guards are separate.
Pruned/failed trials consume attempts but do not count toward successful targets.

The shared runner persists input YAML, histories, reports, sampler state, database,
source identities and worker receipts. Reached targets add no work on rerun.
Deadline cancellations preserve partial evidence without claiming completion.
Unknown crashes remain errors. Source/recipe guards prevent accidental reuse of
another experiment. The unchanged shared TensorBoard callback writes metrics
under the study and confirmation trees; no duplicate training loop is introduced.

In [ ]:
from IPython import get_ipython


TENSORBOARD_DIRECTORY.mkdir(parents=True, exist_ok=True)
get_ipython().run_line_magic("load_ext", "tensorboard")
get_ipython().run_line_magic(
    "tensorboard", 
    f'--logdir "{TENSORBOARD_DIRECTORY}" --port 6006 --bind_all'
)

In [ ]:
display(search_summary(plan))

In [ ]:
first_stage = run_search(
    plan, 
    target_completed=REVIEW_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not first_stage["target_reached"] and not first_stage["time_budget_exhausted"]:
    raise RuntimeError("Attempt ceiling reached before the review target; inspect failed-trial logs.")

## Review the pilot before spending the remaining budget

Inspect completed/pruned counts, width and resampling coverage, failure logs and
duration. The estimate below uses elapsed pilot throughput on **your current
hardware and concurrency**, including pruning overhead. It is a planning estimate:
later TPE suggestions can be larger, and confirmation has performance pruning
disabled. If fewer than two distinct configurations complete, selection cannot
produce two finalists. Do not interpret an incomplete confirmation table as a win.

For a cheaper first look, before first execution choose 12 total hours with a
4-hour confirmation reserve and SEARCH_TARGET=30 in a fresh results directory.
For the broad default space, 24 hours is the more useful starting allowance.

In [ ]:
import optuna


database = Path(plan["study_root"]) / "study.db"
study = None
# Admission or a resumed deadline can prevent the first allocation entirely.
if database.is_file():
    study = optuna.load_study(
        study_name=plan["study_name"], storage="sqlite:///" + database.as_posix()
    )
    trial_table = study.trials_dataframe()
    # No scalar objective exists until at least one trial is allocated.
    if "value" in trial_table.columns:
        display(trial_table.sort_values("value").head(15))
    # Retain useful status columns even before the first score is recorded.
    else:
        display(trial_table)
# A missing database is an unstarted search, not an empty successful study.
else:
    print("No study database yet; inspect admission and budget status.")
display(search_summary(plan))

In [ ]:
from statistics import median


finished = [trial for trial in study.trials if trial.datetime_complete is not None] if study is not None else []
successful = [trial for trial in finished if trial.state.name == "COMPLETE" and trial.value is not None]
# Use the observed wall span, so concurrent trials do not masquerade as serial time.
if successful:
    span_hours = (
        max(trial.datetime_complete for trial in finished)
        - min(trial.datetime_start for trial in finished if trial.datetime_start is not None)
    ).total_seconds() / 3600
    successful_per_hour = len(successful) / max(span_hours, 1e-9)
    typical_trial_hours = median(trial.duration.total_seconds() / 3600 for trial in successful)
    display({
        "observed_successful_trials_per_hour": round(successful_per_hour, 2), 
        "estimated_additional_search_hours": round(max(0, SEARCH_TARGET - len(successful)) / successful_per_hour, 2), 
        "rough_confirmation_hours": round(TOP_K * len(CONFIRMATION_SEEDS) * typical_trial_hours / len(GPU_IDS), 2), 
        "warning": "Pilot extrapolation only; finalist size and early stopping can change runtime."
    })
# A time-limited pilot can end before its first successful result.
else:
    print("No successful pilot trial yet; inspect logs before estimating throughput.")
display(budget_summary(plan))

In [ ]:
main_stage = run_search(
    plan, 
    target_completed=SEARCH_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not main_stage["target_reached"] and not main_stage["time_budget_exhausted"]:
    raise RuntimeError("Attempt ceiling reached before the main target; inspect the search status.")

In [ ]:
# Extra trials use the existing clock and attempt ceiling, never extra hours.
if ENABLE_EXTENSION:
    final_stage = run_search(
        plan, target_completed=EXTENSION_TARGET, max_attempts=MAX_ATTEMPTS, 
        batch_trials=BATCH_TRIALS
    )
    display(final_stage)
# Keep the economical 60-success target when the extension is disabled.
else:
    print("Optional extension disabled.")

## Freeze two finalists and confirm on three paired seeds

The top two distinct finite COMPLETE configurations are frozen with source hashes.
After freezing, search cannot be extended. Each is retrained from scratch with
seeds 101, 202 and 303, keeping the same training/validation split and shuffle seed.
Six runs are outside the search-trial count, with no performance pruning.
Early stopping and the epoch maximum remain. Completed confirmations are reused;
failed/incomplete attempts retain their artifacts and are not successful repeats.

All candidates need every required seed before a fair mean comparison. Three seeds
provide a modest stability check, not proof of optimality. If the shared deadline
prevents completion, the summary marks that limitation explicitly. The recovery
helper below only repairs authenticated orphan records after a stopped search;
it refuses live workers and never restarts completed training.

In [ ]:
from common.dit_hpo_recovery import recover_stopped_search


recovery = recover_stopped_search(plan)
display(recovery)
finalists = freeze_finalists(plan, CONFIRMATION_SEEDS, top_k=TOP_K)
display(finalists)

In [ ]:
confirmation_records = run_confirmations(plan)
print("Completed confirmation runs:", len(confirmation_records))

In [ ]:
import pandas as pd


summary = pd.DataFrame(confirmation_summary(plan))
summary.to_csv(Path(plan["control_root"]) / "confirmation_summary.csv", index=False)
if not summary.empty:
    complete = summary[summary["all_seeds_complete"]]
    partial = summary[~summary["all_seeds_complete"]]
    if not complete.empty:
        print("Configurations with every confirmation seed complete:")
        display(complete.sort_values("mean_noise_loss"))
    if not partial.empty:
        print("Incomplete confirmation progress (unranked). Defer final selection until all candidates finish.")
        display(partial)
else:
    print("No confirmation results are available yet.")
display(budget_summary(plan))


## Simple GPU budget

| Plan | Hardware | Experiment cap | Intended use |
| --- | --- | --- | --- |
| Cheapest first pass | 1 x A100 80 GB | 12 h: 8 search + 4 confirmation | Up to 30 successful trials; broad coverage will be limited |
| Recommended starting allowance | 1 x A100 80 GB | 24 h: 18 search + 6 confirmation | Default target 60 plus six seed confirmations |

These are **spending limits and unbenchmarked planning allowances**, not measured
completion times. The code stops at its persistent deadline even if targets or
confirmations are unfinished. Provider setup, data download, idle rental time,
storage and cleanup are outside the nominal experiment charge. Ending this
notebook does not stop billing or shut down your container.

Runpod's public pricing page viewed 2026-10-10 lists A100 80 GB at **$1.79/GPU-h**:
12 GPU-hours is about **$21.48**; 24 GPU-hours about **$42.96**, excluding extras.
Use your actual quoted rate: `GPUs x hours x price_per_GPU_hour`. Availability,
cloud tier and the live console quote can differ. H100 SXM is listed at $3.99/h;
it needs more than approximately 2.23 times this workload's A100 throughput to
beat that A100 hourly price per completed workload. No UNet speedup is measured
here, so renting several H100s is not the default recommendation.

Sources: [Runpod GPU pricing](https://www.runpod.io/pricing),
[Optuna conditional search spaces](https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/002_configurations.html),
[Optuna PercentilePruner](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.pruners.PercentilePruner.html).

Use the pilot's measured rate to judge whether the desired successful-trial count
fits. More time should follow evidence of useful improvement, not a grid-size
calculation. Keep the official test set untouched until the full recipe is chosen.